In [ ]:
import pandas as pd
import json
import pickle

df_plot = pd.read_csv("./noise_experiment_outputs/df_plot_20260306_123456.csv")

with open("./noise_experiment_outputs/metadata_20260306_123456.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

noise_levels = metadata["noise_levels"]
base_ppl = metadata["base_ppl"]
n_layers = metadata["n_layers"]

with open("./noise_experiment_outputs/results_dict_20260306_123456.pkl", "rb") as f:
    results = pickle.load(f)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.collections import PolyCollection
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import numpy as np

# 보기 좋은 이름 (원하면 수정)
OP_TITLE_SHORT = {
    "ln_1":       "Pre-Attn LN",
    "attn.c_attn":"QKV Proj",
    "attn.c_proj":"Attn Out Proj",
    "ln_2":       "Pre-MLP LN",
    "mlp.c_fc":   "MLP Expand",
    "mlp.c_proj": "MLP Compress",
}

def polygon_under_graph(xlist, ylist):
    return [(xlist[0], 0.0), *zip(xlist, ylist), (xlist[-1], 0.0)]

def plot_critical_layer_module_compare_poly(
    df_plot,
    noise_levels,
    critical_layer_map,   # {"ln_1":3, "attn.c_attn":3, ...}
    use_log=False,        # True면 log1p(ΔPPL)로 스케일 압축
    baseline_noise=None,
    title="Critical-layer module comparison (ΔPPL vs Noise)",
    y_spacing=1.2,        # y축 간격 (겹침 완화)
    legend=True,          # 범례 표시
):
    df = df_plot.copy()
    df["layer"] = df["layer"].astype(int)
    df["noise"] = df["noise"].astype(float)

    noise_levels = sorted([float(n) for n in noise_levels])
    base_noise = float(baseline_noise if baseline_noise is not None else min(noise_levels))

    # x는 index로(그림 스타일 맞추기) 쓰고 tick label만 noise로 표시
    xs = np.arange(len(noise_levels), dtype=float)

    # y축에 표시할 라벨(모듈 + critical layer)
    y_items = []
    ops = list(critical_layer_map.keys())
    for op in ops:
        lyr = critical_layer_map[op]
        name = OP_TITLE_SHORT.get(op, op)
        y_items.append(f"{name} (L{lyr})")

    # y 위치 간격 늘리기
    y_positions = np.arange(len(y_items), dtype=float) * float(y_spacing)

    verts = []
    facecolors = []
    cmap = plt.get_cmap("tab10")

    for idx, op_type in enumerate(ops):
        crit_layer = int(critical_layer_map[op_type])

        d = df[(df["op_type"] == op_type) & (df["layer"] == crit_layer)].copy()

        # layer별 baseline(같은 layer, 같은 op_type에서 noise=base_noise)
        base_series = d[d["noise"] == base_noise]["ppl"]
        if len(base_series) == 0:
            raise ValueError(
                f"[{op_type}, layer {crit_layer}] baseline noise={base_noise} row not found. "
                f"Available noises: {sorted(d['noise'].unique().tolist())}"
            )
        base_ppl = float(base_series.mean())

        zlist = []
        for nz in noise_levels:
            s = d.loc[d["noise"] == nz, "ppl"]
            ppl_val = float(s.iloc[0]) if len(s) else np.nan
            delta = (ppl_val - base_ppl) if np.isfinite(ppl_val) else np.nan

            if use_log:
                delta = max(float(delta), 0.0) if np.isfinite(delta) else 0.0
                delta = np.log1p(delta)  # log(1+Δ)

            zlist.append(float(delta) if np.isfinite(delta) else 0.0)

        verts.append(polygon_under_graph(xs, zlist))
        facecolors.append(cmap(idx % 10))

    fig = plt.figure(figsize=(11, 6))
    ax = fig.add_subplot(111, projection="3d")

    poly = PolyCollection(
        verts, facecolors=facecolors, alpha=0.55,
        edgecolor="k", linewidth=0.6
    )
    ax.add_collection3d(poly, zs=y_positions, zdir="y")

    ax.set_title(
        title + f" | baseline noise={base_noise}" + (" | log1p" if use_log else ""),
        pad=18
    )
    ax.set_xlabel("Noise std")
    ax.set_ylabel("Module index")
    ax.set_zlabel("log1p(ΔPPL)" if use_log else "ΔPPL")

    # X ticks
    ax.set_xlim(xs.min(), xs.max())
    ax.set_xticks(xs)
    ax.set_xticklabels([f"{n:.2f}" for n in noise_levels])

    # Y ticks (중요: ticks 먼저 세팅하고 label을 붙여야 함)
    ax.set_ylim(y_positions.min() - 0.5, y_positions.max() + 0.5)
    ax.set_yticks(y_positions)
    ax.set_yticklabels([str(i) for i in range(len(y_positions))])

    # legend: 모듈명은 legend로 분리해서 겹침 제거
    if legend:
        handles = []
        for idx, label in enumerate(y_items):
            handles.append(Patch(facecolor=cmap(idx % 10), edgecolor="k", label=label, alpha=0.55))

        ax.legend(
            handles=handles,
            loc="upper left",
            bbox_to_anchor=(1.02, 1.0),
            fontsize=9,
            frameon=True
        )

    plt.tight_layout()
    plt.show()


# -------------------------
# 사용 예시
# -------------------------
critical_layer_map = {
    "ln_1": 3,          # Pre-attn LN
    "attn.c_attn": 3,   # QKV
    "attn.c_proj": 0,   # Attn out proj
    "ln_2": 0,          # Pre-MLP LN
    "mlp.c_fc": 1,      # MLP expand
    "mlp.c_proj": 1,    # MLP compress
}

# ΔPPL
plot_critical_layer_module_compare_poly(
    df_plot, noise_levels, critical_layer_map,
    use_log=False,
    title="Critical layer per module — ΔPPL vs Noise",
    y_spacing=1.3,
    legend=True
)

# log1p(ΔPPL)
plot_critical_layer_module_compare_poly(
    df_plot, noise_levels, critical_layer_map,
    use_log=True,
    title="Critical layer per module — ΔPPL vs Noise (log-scaled)",
    y_spacing=1.3,
    legend=True
)


In [ ]:
# =========================================================
# Heatmap (Noise × Layer) with ΔPPL cap = 0.1 * baseline
# - axes unchanged: x=noise (linear), y=layer (0..5)
# - color: capped ΔPPL (NOT log)
# - 2×3 grid for 6 ops + shared colorbar on the right
# =========================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

def plot_layer_capped_delta_ppl_heatmap_grid(
    df_plot,
    noise_levels,
    op_order,
    OP_TITLE,
    baseline_noise=0.0,       # baseline at noise=0 by default
    delta_frac_cap=0.1,       # cap: max ΔPPL = baseline * 0.1
    ncols=3,
    cmap="coolwarm",
    interpolation="bilinear", # smooth gradient: 'bilinear' or 'bicubic'
    eps=1e-12,                # safety for baseline
):
    # ---- style (readability) ----
    plt.rcParams.update({
        "font.size": 13,
        "axes.titlesize": 14,
        "axes.labelsize": 15,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
    })

    # ---- prep ----
    df = df_plot.copy()
    df["layer"] = df["layer"].astype(int)
    df["noise"] = df["noise"].astype(float)
    df = df[df["layer"].between(0, 5)].copy()

    nzs = sorted([float(n) for n in noise_levels])
    layers = list(range(6))

    # aggregate mean ppl per (op, layer, noise)
    agg = (
        df.groupby(["op_type", "layer", "noise"], as_index=False)["ppl"]
        .mean()
        .rename(columns={"ppl": "ppl_mean"})
    )

    # build capped ΔPPL matrices for each op
    Z_list = []
    meta = []  # (layers, nzs, baseline_used)

    for op in op_order:
        a = agg[agg["op_type"] == op].copy()

        # baseline PPL per layer at baseline_noise
        base = (
            a[a["noise"] == float(baseline_noise)]
            .set_index("layer")["ppl_mean"]
            .to_dict()
        )

        Z = np.full((len(layers), len(nzs)), np.nan, dtype=float)

        for yi, layer in enumerate(layers):
            if layer not in base:
                continue
            base_ppl = float(base[layer])
            base_ppl = max(base_ppl, eps)

            cap = base_ppl * float(delta_frac_cap)  # ✅ max ΔPPL

            for xi, nz in enumerate(nzs):
                s = a[(a["layer"] == layer) & (a["noise"] == nz)]["ppl_mean"]
                if len(s) == 0:
                    continue
                ppl = float(s.iloc[0])

                delta = ppl - base_ppl
                # (선택) 음수도 보여주고 싶으면 아래 줄 주석 처리
                delta = max(delta, 0.0)

                # ✅ cap 적용
                Z[yi, xi] = min(delta, cap)

        Z_list.append(Z)
        meta.append((layers, nzs, float(baseline_noise)))

    # ---- shared color scale ----
    all_vals = np.concatenate([z[np.isfinite(z)].ravel() for z in Z_list if np.any(np.isfinite(z))])
    if all_vals.size == 0:
        raise ValueError("No finite values to plot. Check df_plot has matching (op_type, layer, noise, ppl).")

    vmin, vmax = 0.0, float(np.max(all_vals))  # capped already
    norm = Normalize(vmin=vmin, vmax=vmax)

    # ---- grid ----
    n = len(op_order)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.6*ncols, 4.8*nrows))
    axes = np.array(axes).reshape(nrows, ncols)

    im_for_cbar = None
    for i, op in enumerate(op_order):
        r, c = divmod(i, ncols)
        ax = axes[r, c]
        Z = Z_list[i]
        layers, nzs, bnoise = meta[i]

        im = ax.imshow(
            Z,
            aspect="auto",
            origin="lower",
            cmap=cmap,
            norm=norm,
            interpolation=interpolation,  # ✅ smooth gradient
        )
        im_for_cbar = im

        ax.set_title(OP_TITLE.get(op, op), pad=10)
        ax.set_xlabel("Noise")
        ax.set_ylabel("Layer")

        ax.set_xticks(np.arange(len(nzs)))
        ax.set_xticklabels([f"{v:.2f}" for v in nzs], rotation=35, ha="right")
        ax.set_yticks(np.arange(len(layers)))
        ax.set_yticklabels([str(l) for l in layers])

        ax.grid(False)

    # turn off unused axes
    for j in range(n, nrows*ncols):
        r, c = divmod(j, ncols)
        axes[r, c].axis("off")

    # ---- shared colorbar ----
    fig.subplots_adjust(right=0.88, wspace=0.28, hspace=0.38)
    cax = fig.add_axes([0.90, 0.15, 0.02, 0.70])
    cb = fig.colorbar(im_for_cbar, cax=cax)
    cb.set_label(f"ΔPPL (capped at {delta_frac_cap:.2f} × baseline PPL)", fontsize=14)
    cb.ax.tick_params(labelsize=12)

    fig.suptitle(
        f"Smooth Heatmap of Capped ΔPPL (Noise×Layer) — 2×3 Grid | baseline noise={baseline_noise}",
        fontsize=16,
        y=0.98,
    )
    plt.show()


# =========================
# RUN (just copy/paste)
# =========================
plot_layer_capped_delta_ppl_heatmap_grid(
    df_plot=df_plot,
    noise_levels=noise_levels,
    op_order=op_order,
    OP_TITLE=OP_TITLE,
    baseline_noise=0.0,      # change if your baseline is different
    delta_frac_cap=0.1,      # ✅ baseline * 0.1
    ncols=3,
    cmap="viridis",
    interpolation="bicubic" # try 'bicubic' if you want even smoother
)
